In [1]:
from ETGEMs_function_PMI import *
import pandas as pd
import cobra
import ast
from cobra.io import write_sbml_model
from numpy import *
import copy

In [2]:
#Get Reaction G0 from local file_original
reaction_g0_file_original0 = './reaction_g0_strain0.txt'
reaction_g0_file_original1 = './reaction_g0_strain1.txt'
#Get Metabolite concentration from local file
metabolites_lnC_file_original0 = './metabolites_lnC_strain0.txt'
metabolites_lnC_file_original1 = './metabolites_lnC_strain1.txt'
#Get Model from local file
model_file_original0 = './iML1515.xml'
model_file_original1 = './iMM904.xml'
#Get reaction kcat data from ECMpy
reaction_kcat_MW_file_original0 = './ID_kcat_MW_file_strain0.csv'
reaction_kcat_MW_file_original1 = './ID_kcat_MW_file_strain1.csv'

In [3]:
## Convert to usable model formats
model0=Get_Concretemodel_Need_Data(reaction_g0_file_original0,metabolites_lnC_file_original0,model_file_original0,reaction_kcat_MW_file_original0)
model1=Get_Concretemodel_Need_Data(reaction_g0_file_original1,metabolites_lnC_file_original1,model_file_original1,reaction_kcat_MW_file_original1)
## knockout the btuB gene
model0['lb_list']['EX_cbl1_e ']=0
model0['lb_list']['EX_cbl1_e ']=0

In [4]:
# get the information of km, vmax and public metabolites
km = pd.read_csv('./km.csv')
vmax = pd.read_csv('./vmax.csv')
public_metabolism = pd.read_csv('./public_metabolism_test.csv')

'''
public_metabolism_name_list = []
public_metabolism_concentration_list = []
for i in public_metabolism['metabolism']:
    public_metabolism_name_list.append(i)
for j in public_metabolism['concentration']:
    public_metabolism_concentration_list.append(j)
public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
print(public_metabolism)
'''

"\npublic_metabolism_name_list = []\npublic_metabolism_concentration_list = []\nfor i in public_metabolism['metabolism']:\n    public_metabolism_name_list.append(i)\nfor j in public_metabolism['concentration']:\n    public_metabolism_concentration_list.append(j)\npublic_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))\nprint(public_metabolism)\n"

In [5]:
# definate the function to simulate metabolism and interaction
def time_space_state(model_list, biomass_list, growth_list, breed_list, parameter_list, public_metabolism, vmax, km, deta_t, deta_s, micro_distribute_threshold, length, D):
    
    number_cell_side = length // deta_s
    number_cell_side = int(number_cell_side)
    public_metabolism_name_list = []
    public_metabolism_concentration_list = []
    for i in public_metabolism['metabolism']:
        public_metabolism_name_list.append(i)
    for j in public_metabolism['concentration']:
        public_metabolism_concentration_list.append(j)
    public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
    
    number_model = len(model_list)
    
    k_m = {}
    v_max = {}
    for i in range(number_model):
        for j in range(len(km['reactions_strain'+str(i)])):
            k_m[(i, km['reactions_strain'+str(i)][j])] = km['km_strain'+str(i)][j]
            v_max[(i, vmax['reactions_strain'+str(i)][j])] = vmax['vmax_strain'+str(i)][j]
    
    
    number_public_metabolism = len(public_metabolism)
    distribute_micro_list = {}
    distribute_public_metabolism_list = {}
    distribute_lb_list = {}
    public_metabolism_r_list = []
    #set the initial distribution of straints
    for i in range(number_model):
        distribute_micro = np.random.randint(40, size=number_cell_side)
        distribute_micro_list.update({i: distribute_micro})
    print(distribute_micro_list)
    #set the initial distribution of metabolites
    for i in public_metabolism_name_list:
        distribute_public_metabolism = multiply(np.mat(np.ones(number_cell_side)), public_metabolism[i])
        distribute_public_metabolism_list.update({i: distribute_public_metabolism})
    #calculate the upperbounds of uptake reactions for public metabolites
    public_reaction_ub_list = {}
    public_reaction_list = {}
    for i in range(number_model):
        public_reaction_ub = {}
        public_reaction = []
        for rea in model_list[i]['reaction_list']:
            if 'EX_' in rea:
                for j in public_metabolism_name_list:
                    try:
                        model_list[i]['coef_matrix'][(j,rea)]
                    except:
                        pass
                    else:
                        ub = np.mat(np.ones(number_cell_side))
                        if model_list[i]['coef_matrix'][(j,rea)] > 0:
                            for m in range(number_cell_side):
                                ub[0,m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                        else:
                            try:
                                model_list[i]['ub_list'][rea]
                            except:
                                ub = ub * 1000
                            else:
                                ub = ub * model_list[i]['ub_list'][rea]
                        public_reaction_ub.update({rea: ub})
                        public_reaction.append(rea)
                        break
            public_reaction_ub_list[i] = public_reaction_ub
            public_reaction_list[i] = public_reaction

            
    ct = 0
    
    distribute_micro_list_t = {ct: distribute_micro_list}
    distribute_public_metabolism_list_t = {ct: distribute_public_metabolism_list}
    distribute_lb_list_t = {ct: distribute_lb_list}
    r = micro_distribute_threshold + 1
    r_threshold = [r]*5
    slip_r = np.mean(r_threshold[-5:])
    number = {}
    various = {}
    for i in range(number_model):
        number[i] = [np.mean(distribute_micro_list[i])]
        various[i] = [np.std(distribute_micro_list[i])/np.mean(distribute_micro_list[i])]
    
    
    # iterative simulation by slip_r
    while slip_r >= micro_distribute_threshold:
        ct += 1
        print(ct)
        distribute_micro_list_t[ct] = copy.deepcopy(distribute_micro_list_t[ct-1])
        micro_decrease = {}
        micro_increase = {}
        
        #simulate the cell wandering
        #micro_decrease: the decrease number of strains
        #micro_increase: the increase number of strains
        #micro_decrease_cell: the decrease number of strains in the current grid
        #micro_increase_fcell: the increase number of strains in the forward grid
        #micro_increase_bcell: the increase number of strains in the backward grid
        for m in range(number_cell_side):
            micro_decrease_cell = {}
            micro_increase_cell = {}
            for i in range(number_model):
                micro_decrease_cell[i] = 0
                micro_increase_cell[i] = 0
            micro_decrease[m] = micro_decrease_cell
            micro_increase[m] = micro_increase_cell
            
        
        if ct > 0:
            met = 'glc__D_e'
            threshold = 0.3
            for m in range(number_cell_side):
                #calculate the number of strains in the internal grids
                if 0<m<number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = (distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * micro_decrease_cell
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_increase_bcell = micro_decrease_cell-micro_increase_fcell
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                                    micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                                else:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                            elif distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the first grid
                elif m == 0:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the last grid
                elif m == number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_fcell = int(micro_increase_fcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
        
            #calculate the number of strains after wandering
            for m in range(number_cell_side):
                for i in range(number_model):
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] - micro_decrease[m][i]
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] + micro_increase[m][i]
                    distribute_micro_list_t[ct][i][m] = max(0, distribute_micro_list_t[ct][i][m])
                            
                            
        
        #simulate the substrate diffusion by Fick's second law
        for m in range(number_cell_side): 
            if 0<m<number_cell_side-1:
                for met in public_metabolism_name_list:
                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + ((distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s-(distribute_public_metabolism_list[met][0,m]-distribute_public_metabolism_list[met][0,m-1])/deta_s)/deta_s*D*deta_t
                    if distribute_public_metabolism_list[met][0,m] < 0:
                        print('Warning: the D is too high!')
                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == 0:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == number_cell_side-1:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m-1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            
        #simulate the metabolism by ETMs
        biomass_value_list = {}
        number_model_range = []
        for m in range(number_cell_side):
            if m%2 == 0:
                number_model_range.append([0,1])
            elif m%2 == 1:
                number_model_range.append([1,0])
        for m in range(number_cell_side):
            B_value_list = []
            
            
            biomass_value_micro = {}
            for i in number_model_range[m]:
                if distribute_micro_list_t[ct][i][m] > 0:
                    public_metabolism_flux_list = {}
                    for j in public_metabolism_name_list:
                        public_metabolism_flux_list.update({j: 0})
                    for j in public_reaction_list[i]:
                        model_list[i]['ub_list'][j] = int(public_reaction_ub_list[i][j][0,m])
            
                    biomass_id = biomass_list[i]
                    E_total=parameter_list[i][0]
                    #set the carbon source to glucose
                    substrate_name='EX_glc__D_e_reverse'
                    substrate_value=parameter_list[i][1]
                    biomass_value=growth_list[i]
                    K_value=parameter_list[i][2]

                    try:
                        MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                    except:
                        save_rate = 0.9
                        while save_rate >= 0:
                            biomass_value2 = save_rate * biomass_value
                            try:
                                MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                            except:
                                save_rate = save_rate - 0.1
                            else:
                                #calculate the MDF values of metabolic networks
                                biomass_value_micro[i] = biomass_value2
                                B_value2 = MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                                #calculate the biomass yield under the MDF value
                                obj_name=biomass_list[i]
                                obj_target='maximize'
                                if i == 0:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                elif i == 1:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                biomass_value=max_biomass_under_mdf*0.9
                                
                                #calculate the minimum value of the sum of the fluxes
                                if i == 0:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                elif i == 1:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                
                                #translating the results of ETMs into a usable form
                                model=model_list[i]['model']
                                reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                                reaction_g0=model_list[i]['reaction_g0']
                                coef_matrix=model_list[i]['coef_matrix']
                                metabolite_list=model_list[i]['metabolite_list']
                                use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
                                distribute_public_metabolism_increase = {}
                                distribute_public_metabolism_decrease = {}
                                
                                #simulate the fluxes of the public metabolites
                                for rea in public_reaction_list[i]:
                                    for met in public_metabolism_name_list:
                                        try:
                                            model_list[i]['coef_matrix'][(met,rea)]
                                        except:
                                            pass
                                        else:
                                            public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
                                            
                                #simulate the distribution of the public metabolites
                                for met in public_metabolism_name_list:
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]*save_rate
                                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
                                    
                                    
                                save_rate_ub = [save_rate]
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        met = rea[3::]
                                        if met in public_metabolism_name_list:
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            save_rate_met = distribute_public_metabolism_list[met][0,m]/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                            save_rate_ub.append(save_rate_met)
                                        
                                save_rate_final = min(save_rate_ub)
                                
                                
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        if met in public_metabolism_name_list:
                                            met = rea[3::]
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - intracellular_c_dict[met]*int((save_rate_final)*distribute_micro_list_t[ct][i][m]) + intracellular_c_dict[met]*int((1-save_rate_final)*distribute_micro_list_t[ct][i][m])
                                            if distribute_public_metabolism_list[met][0,m] < 0:
                                                print(met + '_3: ', distribute_public_metabolism_list[met][0,m])
                                    
                                
                                distribute_micro_decrease = int((1-save_rate_final) * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_increase = int(save_rate_final * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] - distribute_micro_decrease + distribute_micro_increase)
                                
                                break
                            
                        continue
                    else:
                        #calculate the MDF values of metabolic networks
                        biomass_value_micro[i] = biomass_value
                        B_value=MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                        B_value_list.append(B_value)
                        #calculate the biomass yield under the MDF value
                        obj_name=biomass_list[i]
                        obj_target='maximize'
                        if i == 0:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        biomass_value=max_biomass_under_mdf*0.9
                        
                        #calculate the minimum value of the sum of the fluxes
                        if i == 0:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
            
                        #translating the results of ETMs into a usable form
                        model=model_list[i]['model']
                        reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                        reaction_g0=model_list[i]['reaction_g0']
                        coef_matrix=model_list[i]['coef_matrix']
                        metabolite_list=model_list[i]['metabolite_list']
                        use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
            
                        #simulate the fluxes of the public metabolites
                        for rea in public_reaction_list[i]:
                            for met in public_metabolism_name_list:
                                try:
                                    model_list[i]['coef_matrix'][(met,rea)]  
                                except:
                                    pass
                                else:
                                    public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
            
                        #simulate the distribution of the public metabolites
                        distribute_public_metabolism_ori = copy.deepcopy(distribute_public_metabolism_list)
                        distribute_public_metabolism_re = copy.deepcopy(distribute_public_metabolism_ori)
                        for met in public_metabolism_name_list:
                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]
                            distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                            if distribute_public_metabolism_list[met][0,m] < 0:
                                print(met+'_1: ', distribute_public_metabolism_list[met][0,m])
                                        
                        #simulate the multiplication and death rates of strains and the distribution of public metabolites after multiplication or death        
                        death_rate = 0
                        birth_rate = 1
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    bd_rate_met = (distribute_public_metabolism_list[met][0,m] - 0.1)/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                    if bd_rate_met < 0:
                                        death_rate_lb = (public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] - distribute_public_metabolism_re[met][0,m] + 0.1)/(public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] + intracellular_c_dict[met] * distribute_micro_list_t[ct][i][m])
                                        death_rate = max(death_rate, death_rate_lb)
                                    else:
                                        birth_rate = min(birth_rate, bd_rate_met)
                        death_rate = min(death_rate, 1)
                        birth_rate = min(birth_rate,1)
                        if death_rate > 0:
                            birth_rate = 0
                            for met in public_metabolism_name_list:
                                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + public_metabolism_flux_list[met] * deta_t * math.ceil(distribute_micro_list_t[ct][i][m] * death_rate)
                                distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                        
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_ori[met][0,m] - intracellular_c_dict[met]*math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i]) + intracellular_c_dict[met]*math.ceil(death_rate*distribute_micro_list_t[ct][i][m])
                                    if distribute_public_metabolism_list[met][0,m] < 0:
                                        print(met+'_2: ', distribute_public_metabolism_list[met][0,m])
                                        distribute_public_metabolism_list[met][0,m] = 0
                                
                        #simulate the distribution of strains after multiplication or death
                        distribute_micro_increase = math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i])
                        distribute_micro_decrease = math.ceil(distribute_micro_list_t[ct][i][m]*death_rate)
                        distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] + distribute_micro_increase - distribute_micro_decrease)
                
            #calculate the upperbounds of nutrient uptake rates after substrate diffusion, cell wandering, metabolism, multiplication and death
            biomass_value_list[m] = biomass_value_micro
            
            for i in range(number_model):
                for rea in model_list[i]['reaction_list']:
                    if 'EX_' in rea:
                        for j in public_metabolism_name_list:
                            try:
                                model_list[i]['coef_matrix'][(j,rea)]
                            except:
                                pass
                            else:
                                if model_list[i]['coef_matrix'][(j,rea)] > 0:
                                    public_reaction_ub_list[i][rea][0, m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                                    if i == 0:
                                        # gene upregulation for Fe2+ intake
                                        if 'EX_fe2_e' in rea:
                                            public_reaction_ub_list[i][rea][0, m] = public_reaction_ub_list[i][rea][0, m] * 1.2
        
        # calculate the mean number and uniformity of distribution of strains at this iteration
        for i in range(number_model):
            strain_number = np.mean(distribute_micro_list_t[ct][i])
            strain_various = np.std(distribute_micro_list_t[ct][i])
            number[i].append(strain_number)
            various[i].append(strain_various)
            print('strain_number: ', i, strain_number)
            print('strain_various: ', i, strain_various)
                                    
        # calculate the slip_r at this iteration
        if ct > 1:
            r = 0
            for i in range(number_model):
                for m in range(number_cell_side):
                    if distribute_micro_list_t[ct-1][i][m] > 0:
                        r = r + ((distribute_micro_list_t[ct][i][m]-distribute_micro_list_t[ct-1][i][m])/(distribute_micro_list_t[ct-1][i][m]))**2
                    else:
                        r = r + (distribute_micro_list_t[ct][i][m])**2
        
        r_threshold.append(r)
        slip_r = np.mean(r_threshold[-5:])
        fd_r_threshold = r_threshold[5:]
        print('fd_r_threshold: ', fd_r_threshold)
        print('slip_r: ', slip_r)
    return(distribute_micro_list_t, number, various)

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([29,  2, 30, 15,  6, 38, 32,  7,  1,  0,  2,  9, 21,  4, 29, 33, 32,
       29, 39]), 1: array([27,  8,  9, 26, 24, 27, 22, 18, 13, 29,  8, 18,  0, 34,  2,  0,  2,
       12, 13])}
1
strain_number:  0 22.157894736842106
strain_various:  0 16.49964324301633
strain_number:  1 15.421052631578947
strain_various:  1 10.484654397705972
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 26.210526315789473
strain_various:  0 15.459893604074622
strain_number:  1 15.421052631578947
strain_various:  1 7.036115775607749
fd_r_threshold:  [1.05, 372.3526423360979]
slip_r:  75.31052846721958
3
strain_number:  0 30.94736842105263
strain_various:  0 15.819620343473302
strain_number:  1 15.421052631578947
strain_various:  1 6.293821443737579
fd_r_threshold:  [1.05, 372.3526423360979, 14.94910696330994]
slip_r:  78.09034985988157
4
strain_number:  0 36.8421052631579
strain_various:  0 17.980290840967
strain_number:  1 15.421052631578947
strain_various:  1 5.9144407527916405
fd_r_threshol

glc__D_e_1:  -8.153920830042518
glc__D_e_1:  -3.8838840570752007
glc__D_e_1:  -3.0429569249954276
glc__D_e_1:  -8.016473865422732
glc__D_e_1:  -5.494612276619933
glc__D_e_1:  -2.8708124441192253
glc__D_e_1:  -3.5559726277141666
glc__D_e_1:  -2.6835710979208383
glc__D_e_1:  -11.5743726324703
glc__D_e_1:  -8.332795112766416
glc__D_e_1:  -3.910121912449629
glc__D_e_1:  -4.513097938653747
glc__D_e_1:  -9.764244128059449
glc__D_e_1:  -6.677111030156359
glc__D_e_1:  -5.069608206834811
glc__D_e_1:  -7.63975430770219
glc__D_e_1:  -11.392545383436735
glc__D_e_1:  -8.819641912165363
glc__D_e_1:  -6.7134604461199885
glc__D_e_1:  -5.4048581514299965
glc__D_e_1:  -8.197512213735935
glc__D_e_1:  -5.93349758606615
glc__D_e_1:  -3.3106437532232893
glc__D_e_1:  -4.405412298093223
glc__D_e_1:  -12.46316592551645
glc__D_e_1:  -5.723773741849218
glc__D_e_1:  -1.449052704915754
glc__D_e_1:  -5.4389846080114275
glc__D_e_1:  -7.047370741108704
glc__D_e_1:  -1.5944302930771599
glc__D_e_1:  -0.8884383433250369

glc__D_e_1:  -1.0744146987862537
strain_number:  0 2.6315789473684212
strain_various:  0 0.9296590385608259
strain_number:  1 1.263157894736842
strain_various:  1 0.9647527778854399
fd_r_threshold:  [1.05, 372.3526423360979, 14.94910696330994, 35.392164700559455, 9.539344573000722, 6.570791911565852, 8.347918069946783, 7.494881604027143, 4.603011246228424, 3.93009099976258, 4.851960701618687, 5.999790189536501, 4.3046079352548405, 4.670972876001704, 4.06922669262725, 5.647473594427058, 5.235226757369615, 6.276666666666666]
slip_r:  5.179913317418459
19
glc__D_e_1:  -0.404490034579134
glc__D_e_1:  -0.20783215659171228
glc__D_e_1:  -0.5215562045884153
glc__D_e_1:  -0.7381601735793635
glc__D_e_1:  -0.2512095825045826
glc__D_e_1:  -0.09676516070378849
glc__D_e_1:  -0.08237821502824194
glc__D_e_1:  -0.40443904667697783
glc__D_e_1:  -1.2423331692396629
glc__D_e_1:  -1.7565627958713796
glc__D_e_1:  -0.46966403325672657
glc__D_e_1:  -0.40750480602713446
glc__D_e_1:  -0.2530603842263406
glc__D_

strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 372.3526423360979, 14.94910696330994, 35.392164700559455, 9.539344573000722, 6.570791911565852, 8.347918069946783, 7.494881604027143, 4.603011246228424, 3.93009099976258, 4.851960701618687, 5.999790189536501, 4.3046079352548405, 4.670972876001704, 4.06922669262725, 5.647473594427058, 5.235226757369615, 6.276666666666666, 4.769166666666667, 6.173611111111111, 2.861111111111111, 8.36111111111111, 3.25, 1.25, 0.25, 6.5, 0.0, 1.0, 0.0]
slip_r:  1.55
30
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 372.3526423360979, 14.94910696330994, 35.392164700559455, 9.539344573000722, 6.570791911565852, 8.347918069946783, 7.494881604027143, 4.603011246228424, 3.93009099976258, 4.851960701618687, 5.999790189536501, 4.3046079352548405, 4.670972876001704, 4.06922669

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([18,  0, 37, 16, 24, 20,  1, 22, 20, 20, 15, 19, 30,  8, 10, 30, 16,
       19,  3]), 1: array([34, 12, 12, 12, 27, 26, 18,  2, 10, 32, 34,  2, 37, 31, 24, 16, 18,
       18,  8])}
1
strain_number:  0 20.42105263157895
strain_various:  0 11.430871355439109
strain_number:  1 19.894736842105264
strain_various:  1 11.001888530908563
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 24.157894736842106
strain_various:  0 6.433981806288895
strain_number:  1 19.894736842105264
strain_various:  1 5.839022420361543
fd_r_threshold:  [1.05, 846.0881473108403]
slip_r:  170.05762946216808
3
strain_number:  0 28.526315789473685
strain_various:  0 5.324900997870206
strain_number:  1 19.94736842105263
strain_various:  1 4.882554165603839
fd_r_threshold:  [1.05, 846.0881473108403, 8.790305865011156]
slip_r:  171.6056906351703
4
strain_number:  0 33.78947368421053
strain_various:  0 7.316736023581906
strain_number:  1 20.0
strain_various:  1 4.645314560761311
fd_r_threshold:  [1.05, 84

glc__D_e_1:  -7.515385346164673
glc__D_e_1:  -3.914022621629867
glc__D_e_1:  -0.16056529405650632
glc__D_e_1:  -1.1997420851572884
glc__D_e_1:  -9.962822057536163
glc__D_e_1:  -8.881711104930606
glc__D_e_1:  -2.5527258096061956
glc__D_e_1:  -4.5754876380386085
glc__D_e_1:  -7.678185825006237
glc__D_e_1:  -5.414171197336452
glc__D_e_1:  -2.0764280898132523
glc__D_e_1:  -3.1156048809140344
glc__D_e_1:  -7.945414909123051
glc__D_e_1:  -5.012726233020755
glc__D_e_1:  -2.3840710528100857
glc__D_e_1:  -6.865795474571575
glc__D_e_1:  -7.864971136611568
glc__D_e_1:  -4.932282460509272
glc__D_e_1:  -2.157588965368657
glc__D_e_1:  -2.7049732378036238
glc__D_e_1:  -7.448147938203373
glc__D_e_1:  -4.515459262101078
glc__D_e_1:  -2.0180355758587574
glc__D_e_1:  -4.532589922956985
glc__D_e_1:  -11.288506759685085
glc__D_e_1:  -8.71560328841371
glc__D_e_1:  -2.017789826093858
glc__D_e_1:  -4.532344173192085
glc__D_e_1:  -7.268983000696663
glc__D_e_1:  -5.8280868432601824
glc__D_e_1:  -2.9799526009980

glc__D_e_1:  -0.2407380392019851
glc__D_e_1:  -0.7549676658337017
glc__D_e_1:  -0.9268201166870385
glc__D_e_1:  -1.0921711536676229
glc__D_e_1:  -0.1064045511896663
glc__D_e_1:  -1.9090010046995751
glc__D_e_1:  -1.6001121610979872
glc__D_e_1:  -0.6231846668116703
glc__D_e_1:  -0.2788782508104375
glc__D_e_1:  -0.12443382900964339
glc__D_e_1:  -0.7154011745193791
glc__D_e_1:  -2.5423290484308003
glc__D_e_1:  -1.5647661563967017
glc__D_e_1:  -1.0834821242615948
glc__D_e_1:  -2.1497534552892765
glc__D_e_1:  -1.8408646116876886
strain_number:  0 2.263157894736842
strain_various:  0 1.1164843913471802
strain_number:  1 1.7894736842105263
strain_various:  1 0.6942582083301537
fd_r_threshold:  [1.05, 846.0881473108403, 8.790305865011156, 7.4297885701588084, 7.361351947646804, 7.647217431214965, 9.132713149626781, 6.9905891566407075, 4.599107500785354, 5.107925354843251, 5.17828965901022, 4.144858541405281, 4.518567721363086, 4.821831341178697, 2.90510426362158, 3.7258234126984133, 5.1675314940

glc__D_e_1:  -0.2215004576078175
glc__D_e_1:  -0.023790490530757713
strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 846.0881473108403, 8.790305865011156, 7.4297885701588084, 7.361351947646804, 7.647217431214965, 9.132713149626781, 6.9905891566407075, 4.599107500785354, 5.107925354843251, 5.17828965901022, 4.144858541405281, 4.518567721363086, 4.821831341178697, 2.90510426362158, 3.7258234126984133, 5.167531494079113, 4.148021541950114, 6.326666666666666, 3.173611111111111, 6.861111111111111, 4.25, 5.75, 3.0, 3.5, 2.0, 0.0, 4.0]
slip_r:  2.5
29
glc__D_e_1:  -0.02159305951094259
glc__D_e_1:  -0.04761778989293308
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 846.0881473108403, 8.790305865011156, 7.4297885701588084, 7.361351947646804, 7.647217431214965, 9.13271314962

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 6, 24,  7, 23, 14, 12,  0, 30, 27, 33, 11, 32,  9,  0, 18, 15, 20,
       25, 26]), 1: array([29, 20, 22, 12, 27, 11,  3,  9, 22, 27, 18, 17, 25, 38, 19,  6, 19,
       16, 27])}
1
strain_number:  0 20.63157894736842
strain_various:  0 12.022831465769352
strain_number:  1 19.36842105263158
strain_various:  1 8.572971764525196
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 24.31578947368421
strain_various:  0 9.690215529507599
strain_number:  1 19.36842105263158
strain_various:  1 5.352918793722577
fd_r_threshold:  [1.05, 272.85941496969605]
slip_r:  55.41188299393921
3
strain_number:  0 28.789473684210527
strain_various:  0 8.507451516284949
strain_number:  1 19.36842105263158
strain_various:  1 4.70397957294828
fd_r_threshold:  [1.05, 272.85941496969605, 202.21211124272938]
slip_r:  95.64430524248507
4
strain_number:  0 34.0
strain_various:  0 8.795932074125606
strain_number:  1 19.36842105263158
strain_various:  1 4.933350802294942
fd_r_threshold:  [1.05, 272.8

glc__D_e_1:  -7.04784082464273
glc__D_e_1:  -4.938270618773739
glc__D_e_1:  -5.011230252333247
glc__D_e_1:  -6.597791315868995
glc__D_e_1:  -10.11595539756348
glc__D_e_1:  -8.366170396525412
glc__D_e_1:  -6.463276311354274
glc__D_e_1:  -10.508799968219101
glc__D_e_1:  -7.309099767292637
glc__D_e_1:  -6.536877658288667
glc__D_e_1:  -2.45727504626562
glc__D_e_1:  -2.512866800034771
glc__D_e_1:  -13.757046190134428
glc__D_e_1:  -7.531883633098914
glc__D_e_1:  -0.13948979540442097
glc__D_e_1:  -1.1786665865052033
glc__D_e_1:  -11.082962689517128
glc__D_e_1:  -7.1727111213807335
glc__D_e_1:  -3.738091391510273
glc__D_e_1:  -2.8656898617169446
glc__D_e_1:  -4.700113021251153
glc__D_e_1:  -3.4136612856154662
glc__D_e_1:  -5.465622815085608
glc__D_e_1:  -7.543976397287172
glc__D_e_1:  -5.963476554161376
glc__D_e_1:  -4.522580396724896
glc__D_e_1:  -2.416357968809203
glc__D_e_1:  -2.96374224124417
glc__D_e_1:  -9.21244645060192
glc__D_e_1:  -7.462661449563852
glc__D_e_1:  -2.7591247024472905
gl

glc__D_e_1:  -2.3491085571654255
glc__D_e_1:  -1.4704721690256735
glc__D_e_1:  -0.9243300750043013
glc__D_e_1:  -0.5393475457297572
glc__D_e_1:  -0.3849031239289632
glc__D_e_1:  -0.700631127219709
glc__D_e_1:  -0.38764372625296284
glc__D_e_1:  -0.268214280515944
glc__D_e_1:  -0.49260294140053773
glc__D_e_1:  -1.2934085744293582
glc__D_e_1:  -0.13495179429020965
glc__D_e_1:  -0.6491814209219262
strain_number:  0 2.526315789473684
strain_various:  0 0.8187552203212656
strain_number:  1 1.5789473684210527
strain_various:  1 0.8153649149910351
fd_r_threshold:  [1.05, 272.85941496969605, 202.21211124272938, 9.544108715981952, 10.364178734369727, 10.175543277677015, 8.054768884233038, 6.038159142820774, 4.256842788318023, 4.3415819825274236, 4.557821177095357, 4.643772485655914, 4.74443870557184, 3.3545685366471343, 3.3575872898469057, 5.6417977145709814, 4.7925, 7.311666666666666]
slip_r:  4.891624041546338
19
glc__D_e_1:  -0.11889447358931449
glc__D_e_1:  -1.0523573360467653
glc__D_e_1:  -

glc__D_e_1:  -0.030802116197121476
strain_number:  0 0.3157894736842105
strain_various:  0 0.464829519280413
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 272.85941496969605, 202.21211124272938, 9.544108715981952, 10.364178734369727, 10.175543277677015, 8.054768884233038, 6.038159142820774, 4.256842788318023, 4.3415819825274236, 4.557821177095357, 4.643772485655914, 4.74443870557184, 3.3545685366471343, 3.3575872898469057, 5.6417977145709814, 4.7925, 7.311666666666666, 5.206666666666667, 3.8958333333333335, 5.833333333333334, 5.25, 3.25, 7.5, 2.0, 1.0, 2.0, 1.0, 2.0]
slip_r:  1.6
30
glc__D_e_1:  -0.08810052882031921
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 272.85941496969605, 202.21211124272938, 9.544108715981952, 10.364178734369727, 10.175543277677015, 8.054768884233038, 6.038159142820774, 4.256842788318023, 4.3415819825274236, 4.557821177095357, 4.64377248565

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([30,  0, 31,  4, 22, 13, 17, 16,  1, 12, 14, 28, 10, 30, 24, 35,  6,
       32, 30]), 1: array([18, 26, 13,  5,  6,  2, 25, 38, 24, 23,  2, 16, 39, 32,  6, 14, 33,
       28,  1])}
1
strain_number:  0 22.105263157894736
strain_various:  0 13.333356417339205
strain_number:  1 18.68421052631579
strain_various:  1 12.511269158469842
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 26.210526315789473
strain_various:  0 9.099328762799924
strain_number:  1 18.789473684210527
strain_various:  1 8.256282758369506
fd_r_threshold:  [1.05, 833.3757341571667]
slip_r:  167.51514683143336
3
strain_number:  0 31.105263157894736
strain_various:  0 10.745352664204015
strain_number:  1 18.789473684210527
strain_various:  1 4.752605350513393
fd_r_threshold:  [1.05, 833.3757341571667, 11.370494165872053]
slip_r:  169.57924566460775
4
strain_number:  0 37.0
strain_various:  0 12.884098726725126
strain_number:  1 18.789473684210527
strain_various:  1 4.640541567401499
fd_r_threshold:  [1.

glc__D_e_1:  -5.3635396370775785
strain_number:  0 16.31578947368421
strain_various:  0 7.108576225636093
strain_number:  1 7.894736842105263
strain_various:  1 3.354411656658567
fd_r_threshold:  [1.05, 833.3757341571667, 11.370494165872053, 7.621221676683632, 10.257469071408778, 7.3815826029452625, 7.783778833636843, 7.02716508976248, 4.26012721304352, 4.849802024749348, 4.233230284168319, 7.235658250815196]
slip_r:  5.5211965725077725
13
glc__D_e_1:  -7.634439420279811
glc__D_e_1:  -2.6957285988799837
glc__D_e_1:  -0.40928316974218015
glc__D_e_1:  -3.415630035506223
glc__D_e_1:  -13.023895945593202
glc__D_e_1:  -8.290525907223502
glc__D_e_1:  -0.5302095046888289
glc__D_e_1:  -0.5858012584579804
glc__D_e_1:  -12.489415380476046
glc__D_e_1:  -7.0873712936738364
glc__D_e_1:  -6.456187565545898
glc__D_e_1:  -10.009918703744908
glc__D_e_1:  -8.23661812536283
glc__D_e_1:  -5.303929449260534
glc__D_e_1:  -4.003849468939681
glc__D_e_1:  -3.6232404578121677
glc__D_e_1:  -8.52958869613303
glc_

glc__D_e_1:  -1.7350163574690889
glc__D_e_1:  -1.0397134055248047
glc__D_e_1:  -0.8852689837240107
glc__D_e_1:  -1.2618859243511789
glc__D_e_1:  -1.990468880479749
glc__D_e_1:  -0.3442319400131397
glc__D_e_1:  -0.8207214860069394
glc__D_e_1:  -1.2322344850753264
glc__D_e_1:  -0.1525631901392428
glc__D_e_1:  -1.4849716767501904
glc__D_e_1:  -0.5074087847160917
glc__D_e_1:  -1.2301579939733758
glc__D_e_1:  -0.7616398315935771
glc__D_e_1:  -1.124516704321556
glc__D_e_1:  -0.28442804791003073
glc__D_e_1:  -0.7986576745417473
glc__D_e_1:  -0.7002222500263466
glc__D_e_1:  -0.9593445447639699
glc__D_e_1:  -0.7478514369797173
glc__D_e_1:  -0.3586869344323138
glc__D_e_1:  -0.3503626065526424
glc__D_e_1:  -1.247089766087969
glc__D_e_1:  -1.0926453442871749
strain_number:  0 2.6842105263157894
strain_various:  0 1.0286747518785297
strain_number:  1 1.263157894736842
strain_various:  1 0.5469634129164876
fd_r_threshold:  [1.05, 833.3757341571667, 11.370494165872053, 7.621221676683632, 10.257469071

glc__D_e_1:  -0.05215028078351275
strain_number:  0 0.3684210526315789
strain_various:  0 0.48237638894272
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 833.3757341571667, 11.370494165872053, 7.621221676683632, 10.257469071408778, 7.3815826029452625, 7.783778833636843, 7.02716508976248, 4.26012721304352, 4.849802024749348, 4.233230284168319, 7.235658250815196, 4.262683929349903, 4.807160443318788, 4.116990319577317, 3.2944496097337015, 4.917711167800453, 5.573117913832199, 6.109444444444445, 6.423611111111111, 4.333333333333334, 6.611111111111111, 3.5, 1.25, 6.0, 2.0, 0.0, 1.0]
slip_r:  2.05
29
strain_number:  0 0.3684210526315789
strain_various:  0 0.48237638894272
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 833.3757341571667, 11.370494165872053, 7.621221676683632, 10.257469071408778, 7.3815826029452625, 7.783778833636843, 7.02716508976248, 4.26012721304352, 4.849802024749348, 4.233230284168319, 7.235658250815196, 4.262683929349903, 4.80

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 2,  0, 16, 13, 25,  0, 21, 37, 10, 32,  1,  3,  6, 26, 13, 23, 21,
        6, 33]), 1: array([37, 11, 10, 34, 14,  8,  6,  4, 29, 25,  7, 19, 20, 18, 32, 39, 35,
        0,  0])}
1
strain_number:  0 17.894736842105264
strain_various:  0 13.999604268249515
strain_number:  1 18.57894736842105
strain_various:  1 13.035642594031764
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 21.05263157894737
strain_various:  0 9.827317909038994
strain_number:  1 18.68421052631579
strain_various:  1 8.903297033174947
fd_r_threshold:  [1.05, 689.9173206030356]
slip_r:  138.82346412060713
3
strain_number:  0 24.789473684210527
strain_various:  0 9.105111042234068
strain_number:  1 18.736842105263158
strain_various:  1 7.275352082369462
fd_r_threshold:  [1.05, 689.9173206030356, 8.635041891503782]
slip_r:  140.34047249890787
4
strain_number:  0 29.31578947368421
strain_various:  0 9.776731945523823
strain_number:  1 18.736842105263158
strain_various:  1 5.954583420518295
fd_r_thresho

glc__D_e_1:  -6.9994825258495945
glc__D_e_1:  -4.221238271548092
glc__D_e_1:  -4.18333804542022
glc__D_e_1:  -7.737069183619229
glc__D_e_1:  -10.660888138560985
glc__D_e_1:  -4.744614425127058
glc__D_e_1:  -2.896578604052512
glc__D_e_1:  -8.361888063145631
glc__D_e_1:  -8.39028124065755
glc__D_e_1:  -6.794940661420276
glc__D_e_1:  -2.3060588555240695
glc__D_e_1:  -2.853443127959036
glc__D_e_1:  -12.238029948154848
glc__D_e_1:  -8.84200800665017
glc__D_e_1:  -3.6987896311741792
glc__D_e_1:  -1.842803064049221
glc__D_e_1:  -6.126849402569933
glc__D_e_1:  -5.354627293565963
glc__D_e_1:  -5.0328037532338685
glc__D_e_1:  -6.619364816769616
glc__D_e_1:  -4.140342913405524
glc__D_e_1:  -3.0083355995706316
glc__D_e_1:  -5.8042569478653165
glc__D_e_1:  -8.866195567398512
glc__D_e_1:  -6.609992757319627
glc__D_e_1:  -3.831748503018124
strain_number:  0 13.526315789473685
strain_various:  0 4.848965392118324
strain_number:  1 7.526315789473684
strain_various:  1 2.8352744423485534
fd_r_threshold:

glc__D_e_1:  -1.045346165001791
glc__D_e_1:  -1.5595757916335078
glc__D_e_1:  -0.8951730797699102
glc__D_e_1:  -2.2053979941770754
glc__D_e_1:  -1.2278351021429768
glc__D_e_1:  -0.5788839446934737
glc__D_e_1:  -0.09970272192975083
glc__D_e_1:  -1.1501506122460632
glc__D_e_1:  -0.5320702364977752
strain_number:  0 3.210526315789474
strain_various:  0 1.0552598766191275
strain_number:  1 1.631578947368421
strain_various:  1 0.6657426652986062
fd_r_threshold:  [1.05, 689.9173206030356, 8.635041891503782, 30.52831871266496, 9.285270321600876, 13.016931231025753, 14.846219464500006, 9.756594555027267, 8.809455415781528, 5.535866940246206, 5.32158434118955, 3.989336449213254, 4.348038946607785, 4.288015749035563, 7.589991964812976, 3.495089965049381, 4.457908163265307, 4.066944444444445]
slip_r:  4.779590057321535
19
glc__D_e_1:  -0.43774033215210517
glc__D_e_1:  -0.9519699587838217
glc__D_e_1:  -0.4543842609686526
glc__D_e_1:  -1.880140156995089
glc__D_e_1:  -1.2551868083228899
glc__D_e_1: 

glc__D_e_1:  -0.23838076686034232
glc__D_e_1:  -0.29457842209542406
glc__D_e_1:  -0.14184192539861207
glc__D_e_1:  -0.05479083764060316
strain_number:  0 0.10526315789473684
strain_various:  0 0.30689220499185793
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 689.9173206030356, 8.635041891503782, 30.52831871266496, 9.285270321600876, 13.016931231025753, 14.846219464500006, 9.756594555027267, 8.809455415781528, 5.535866940246206, 5.32158434118955, 3.989336449213254, 4.348038946607785, 4.288015749035563, 7.589991964812976, 3.495089965049381, 4.457908163265307, 4.066944444444445, 4.970555555555555, 6.583333333333334, 7.083333333333334, 6.722222222222222, 6.0625, 0.6111111111111112, 1.0, 3.0, 5.0, 6.0]
slip_r:  3.1222222222222222
29
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 689.9173206030356, 8.635041891503782, 30.52831871266496, 9.285270321600876, 13.01693123102575

In [9]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([25, 27,  1, 27, 24,  2, 10, 18,  5, 36, 30, 36,  0,  2, 35, 30, 31,
       36, 18]), 1: array([21, 16, 13, 23, 27, 25, 23,  3, 35,  4, 21, 39, 16, 29,  4,  2, 10,
       11, 17])}
1
strain_number:  0 24.57894736842105
strain_various:  0 15.554034133926
strain_number:  1 17.94736842105263
strain_various:  1 10.63001551818705
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 29.105263157894736
strain_various:  0 11.629792670650962
strain_number:  1 18.0
strain_various:  1 7.595012213704188
fd_r_threshold:  [1.05, 752.2368528859041]
slip_r:  151.28737057718084
3
strain_number:  0 34.526315789473685
strain_various:  0 9.91738728885381
strain_number:  1 18.0
strain_various:  1 6.407232755171875
fd_r_threshold:  [1.05, 752.2368528859041, 9.063542725886977]
slip_r:  152.89007912235823
4
strain_number:  0 41.0
strain_various:  0 11.639316311988704
strain_number:  1 18.0
strain_various:  1 4.963021151211066
fd_r_threshold:  [1.05, 752.2368528859041, 9.063542725886977, 6.45895

glc__D_e_1:  -10.743123070391485
glc__D_e_1:  -5.4955234053900694
glc__D_e_1:  -4.920402061115654
glc__D_e_1:  -8.474133199314664
glc__D_e_1:  -10.171623237888287
glc__D_e_1:  -4.409793946255155
glc__D_e_1:  -1.407781807398487
glc__D_e_1:  -3.4305436358309
glc__D_e_1:  -13.98443092397078
glc__D_e_1:  -5.753246221637735
glc__D_e_1:  -1.041675227380114
glc__D_e_1:  -4.539814611809972
glc__D_e_1:  -9.564413707784439
glc__D_e_1:  -2.4652363192862854
strain_number:  0 14.842105263157896
strain_various:  0 6.149567018517276
strain_number:  1 6.7368421052631575
strain_various:  1 3.2741749505254294
fd_r_threshold:  [1.05, 752.2368528859041, 9.063542725886977, 6.4589581221123185, 9.031914396034345, 12.744685507428196, 7.409850180890386, 5.844460948965159, 3.9472526645704393, 4.193852032971931, 6.003897909241402, 4.56602103490331]
slip_r:  4.911096918130449
13
glc__D_e_1:  -6.349871683714961
glc__D_e_1:  -2.9029533809809487
glc__D_e_1:  -1.7097211044122054
glc__D_e_1:  -2.2571053768471723
glc__

glc__D_e_1:  -0.5285979244679178
glc__D_e_1:  -0.3138453317854537
glc__D_e_1:  -0.2065635707145046
glc__D_e_1:  -0.0521191489137105
glc__D_e_1:  -0.480489062202909
glc__D_e_1:  -0.3327692563698892
glc__D_e_1:  -0.8469988830016058
glc__D_e_1:  -2.108970181042207
glc__D_e_1:  -0.7725678389175936
glc__D_e_1:  -1.2867974655493102
glc__D_e_1:  -0.30198384003322887
glc__D_e_1:  -0.6826622169157909
glc__D_e_1:  -1.1616753526127555
glc__D_e_1:  -1.0072309308119616
glc__D_e_1:  -0.8521131275994372
glc__D_e_1:  -0.11568047996023911
glc__D_e_1:  -0.5103644505671228
glc__D_e_1:  -0.26766263200631846
glc__D_e_1:  -0.3796763303646453
glc__D_e_1:  -0.6930496431343645
strain_number:  0 2.6842105263157894
strain_various:  0 1.2999041089714178
strain_number:  1 0.8947368421052632
strain_various:  1 0.6402907926629705
fd_r_threshold:  [1.05, 752.2368528859041, 9.063542725886977, 6.4589581221123185, 9.031914396034345, 12.744685507428196, 7.409850180890386, 5.844460948965159, 3.9472526645704393, 4.19385203

strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 752.2368528859041, 9.063542725886977, 6.4589581221123185, 9.031914396034345, 12.744685507428196, 7.409850180890386, 5.844460948965159, 3.9472526645704393, 4.193852032971931, 6.003897909241402, 4.56602103490331, 5.471908639817787, 3.8141591988397274, 5.313800674233574, 3.8181434240362817, 5.918151927437642, 5.457222222222223, 3.0700000000000003, 8.776666666666667, 7.111111111111111, 1.6111111111111112, 6.0, 2.75, 3.25, 1.0, 4.0, 1.0]
slip_r:  2.4
29
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 752.2368528859041, 9.063542725886977, 6.4589581221123185, 9.031914396034345, 12.744685507428196, 7.409850180890386, 5.844460948965159, 3.9472526645704393, 4.193852032971931, 6.003897909241402, 4.56602103490331, 5.471908639817787, 3.8141591988397274, 5.313800674233574, 3.8181434240362817,

In [10]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([21, 34, 35, 37, 37,  9,  5,  8, 12, 23, 37, 25, 11, 10, 24, 36, 38,
       12, 11]), 1: array([ 9, 36, 12, 22, 25,  7, 25, 12, 37, 16, 26, 16, 26, 26,  3, 23,  8,
       37, 27])}
1
strain_number:  0 26.473684210526315
strain_various:  0 14.251014446080049
strain_number:  1 20.842105263157894
strain_various:  1 10.327597967504028
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 31.42105263157895
strain_various:  0 11.811304974800889
strain_number:  1 20.842105263157894
strain_various:  1 4.380136478223828
fd_r_threshold:  [1.05, 28.723839304705013]
slip_r:  6.584767860941002
3
strain_number:  0 37.26315789473684
strain_various:  0 11.087413942053288
strain_number:  1 20.894736842105264
strain_various:  1 4.302933354940988
fd_r_threshold:  [1.05, 28.723839304705013, 7.699589262195697]
slip_r:  7.914685713380142
4
strain_number:  0 44.36842105263158
strain_various:  0 11.305380769749883
strain_number:  1 20.894736842105264
strain_various:  1 3.9986147185731746
fd_r_th

glc__D_e_1:  -8.508864960472554
glc__D_e_1:  -4.481589767959328
glc__D_e_1:  -1.8577899354586198
glc__D_e_1:  -2.745382606465558
glc__D_e_1:  -7.2271070282270475
glc__D_e_1:  -5.995815876941977
glc__D_e_1:  -4.554919719505497
glc__D_e_1:  -7.436124536430652
glc__D_e_1:  -9.078277353735551
glc__D_e_1:  -6.233922330816228
glc__D_e_1:  -4.793026173379747
glc__D_e_1:  -3.1028577768334404
glc__D_e_1:  -5.617412123931668
glc__D_e_1:  -10.796838240142364
glc__D_e_1:  -10.229956914168522
glc__D_e_1:  -5.295224925873235
glc__D_e_1:  -5.898200952077353
glc__D_e_1:  -4.648103597488131
glc__D_e_1:  -4.0303259102849545
glc__D_e_1:  -6.640023499817983
glc__D_e_1:  -9.701962119351178
glc__D_e_1:  -7.698535239860238
glc__D_e_1:  -5.434520612190453
glc__D_e_1:  -1.8367818478614124
glc__D_e_1:  -3.36775115762801
glc__D_e_1:  -12.065944416169213
glc__D_e_1:  -5.995226280934492
glc__D_e_1:  -1.5344479720825057
glc__D_e_1:  -3.5572098005149186
glc__D_e_1:  -12.576734043531673
glc__D_e_1:  -7.84336400516197

glc__D_e_1:  -0.8485613363875149
glc__D_e_1:  -0.5681909520772286
glc__D_e_1:  -0.45881265982147323
glc__D_e_1:  -0.30436823802067936
glc__D_e_1:  -1.2711653288690483
glc__D_e_1:  -0.4207655473277594
glc__D_e_1:  -1.0008143649259607
glc__D_e_1:  -0.07282108136348153
glc__D_e_1:  -2.0620772573662935
glc__D_e_1:  -1.084514365332195
strain_number:  0 2.5789473684210527
strain_various:  0 0.9902572485382516
strain_number:  1 1.7894736842105263
strain_various:  1 0.6942582083301536
fd_r_threshold:  [1.05, 28.723839304705013, 7.699589262195697, 7.327118287808103, 9.035449806883978, 7.105043898262534, 5.047827798152083, 3.4738054208719173, 4.837815025948059, 4.340079620883162, 4.649397255363635, 5.228006731678601, 4.377945487374672, 4.638505819908375, 3.7892907533383724, 4.63875283446712, 4.965833333333332]
slip_r:  4.482065645684374
18
glc__D_e_1:  -0.7783652025749617
glc__D_e_1:  -0.519104859790422
glc__D_e_1:  -0.7171059653399765
glc__D_e_1:  -0.8508713791855755
glc__D_e_1:  -0.46550395569

strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 28.723839304705013, 7.699589262195697, 7.327118287808103, 9.035449806883978, 7.105043898262534, 5.047827798152083, 3.4738054208719173, 4.837815025948059, 4.340079620883162, 4.649397255363635, 5.228006731678601, 4.377945487374672, 4.638505819908375, 3.7892907533383724, 4.63875283446712, 4.965833333333332, 5.006944444444444, 3.9722222222222223, 5.5625, 3.0, 6.361111111111111, 3.0, 6.5, 3.0, 2.0, 3.0, 0.0]
slip_r:  2.9
29
glc__D_e_1:  -0.10357381736606996
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 28.723839304705013, 7.699589262195697, 7.327118287808103, 9.035449806883978, 7.105043898262534, 5.047827798152083, 3.4738054208719173, 4.837815025948059, 4.340079620883162, 4.649397255363

In [11]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([32,  1, 11, 13, 21, 19, 12,  5, 35, 17, 37, 26, 26, 35, 39, 10, 25,
       35, 32]), 1: array([33, 22, 23,  8, 16, 35,  2,  8,  7,  1, 27, 31, 11, 12,  8,  9, 23,
       39,  3])}
1
strain_number:  0 26.94736842105263
strain_various:  0 13.643867657440085
strain_number:  1 16.94736842105263
strain_various:  1 12.023983420036746
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.05263157894737
strain_various:  0 10.995088301555507
strain_number:  1 17.0
strain_various:  1 7.20380016672487
fd_r_threshold:  [1.05, 456.05898503296225]
slip_r:  92.05179700659245
3
strain_number:  0 38.05263157894737
strain_various:  0 10.937495548079822
strain_number:  1 17.0
strain_various:  1 4.952405051030629
fd_r_threshold:  [1.05, 456.05898503296225, 6.2251898920246225]
slip_r:  93.08683498499737
4
strain_number:  0 45.36842105263158
strain_various:  0 14.3130802389974
strain_number:  1 17.0
strain_various:  1 4.388981418818818
fd_r_threshold:  [1.05, 456.05898503296225, 6.22518989

glc__D_e_1:  -7.644627010477999
glc__D_e_1:  -19.604423648931
glc__D_e_1:  -6.074742920367203
glc__D_e_1:  -1.549689140069903
glc__D_e_1:  -3.572450968502316
glc__D_e_1:  -7.517224039273504
glc__D_e_1:  -2.5785132178736765
glc__D_e_1:  -2.119499318800934
glc__D_e_1:  -5.125846184564977
glc__D_e_1:  -8.576312596917225
glc__D_e_1:  -4.306275823949908
glc__D_e_1:  -3.1839488372808473
glc__D_e_1:  -7.1738807403765215
glc__D_e_1:  -9.387891710908765
glc__D_e_1:  -2.9573883708431215
glc__D_e_1:  -1.310794446596835
glc__D_e_1:  -2.8417637563634326
glc__D_e_1:  -9.077724954490574
glc__D_e_1:  -4.6532437597224625
glc__D_e_1:  -1.966698319024955
glc__D_e_1:  -6.940215259452259
glc__D_e_1:  -4.99575609325839
glc__D_e_1:  -3.040630309190193
strain_number:  0 14.526315789473685
strain_various:  0 5.660281014528776
strain_number:  1 6.157894736842105
strain_various:  1 2.795921049535458
fd_r_threshold:  [1.05, 456.05898503296225, 6.2251898920246225, 5.219408301628773, 5.776654770858721, 6.1910848529

glc__D_e_1:  -0.3416956830633002
glc__D_e_1:  -0.18725126126250613
glc__D_e_1:  -1.0549220437120645
glc__D_e_1:  -0.568804076997881
strain_number:  0 3.210526315789474
strain_various:  0 1.0041465278073112
strain_number:  1 1.2105263157894737
strain_various:  1 0.6942582083301536
fd_r_threshold:  [1.05, 456.05898503296225, 6.2251898920246225, 5.219408301628773, 5.776654770858721, 6.191084852902252, 4.608119949940729, 4.9813540982947595, 4.447152035597948, 5.184096124860039, 3.979740919950094, 5.830025929044272, 3.724269458349951, 4.32113474515632, 4.034397536486281, 3.8795090984052027, 7.516697530864198]
slip_r:  4.695201673852391
18
glc__D_e_1:  -1.190287612689857
glc__D_e_1:  -1.2545656403645822
glc__D_e_1:  -0.4960748377041837
glc__D_e_1:  -1.0103044643359005
glc__D_e_1:  -0.5908186489541656
glc__D_e_1:  -1.3112822722687985
glc__D_e_1:  -1.631382794906405
glc__D_e_1:  -0.6686786480861329
glc__D_e_1:  -0.39875707161307017
glc__D_e_1:  -0.0891476602685346
glc__D_e_1:  -0.2323815356848

strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 456.05898503296225, 6.2251898920246225, 5.219408301628773, 5.776654770858721, 6.191084852902252, 4.608119949940729, 4.9813540982947595, 4.447152035597948, 5.184096124860039, 3.979740919950094, 5.830025929044272, 3.724269458349951, 4.32113474515632, 4.034397536486281, 3.8795090984052027, 7.516697530864198, 4.129444444444445, 5.340277777777778, 8.86111111111111, 3.361111111111111, 4.0, 2.5, 6.25, 1.0, 3.0, 1.0, 0.0]
slip_r:  2.25
29
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 456.05898503296225, 6.2251898920246225, 5.219408301628773, 5.776654770858721, 6.191084852902252, 4.608119949940729, 4.9813540982947595, 4.447152035597948, 5.184096124860039, 3.979740919950094, 5.830025929044272, 3.724269458349951, 4.32113474515632, 4.034397536486281, 3.87950

In [12]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([39,  9, 28, 11, 24,  0,  8, 28, 30, 30, 16,  9,  5, 20, 19, 25, 35,
       35, 38]), 1: array([39, 31, 20, 13,  7, 33,  5, 12,  4, 30, 38, 11, 25, 11, 39, 14, 39,
       35, 28])}
1
strain_number:  0 25.473684210526315
strain_various:  0 13.95620375563133
strain_number:  1 23.263157894736842
strain_various:  1 12.826133294600472
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.157894736842106
strain_various:  0 12.42774128563207
strain_number:  1 23.42105263157895
strain_various:  1 8.512008789281696
fd_r_threshold:  [1.05, 31.887449031454906]
slip_r:  7.217489806290982
3
strain_number:  0 35.78947368421053
strain_various:  0 12.767470785136243
strain_number:  1 23.57894736842105
strain_various:  1 7.117922445812285
fd_r_threshold:  [1.05, 31.887449031454906, 72.77949791254017]
slip_r:  21.563389388799017
4
strain_number:  0 42.526315789473685
strain_various:  0 15.32925795935627
strain_number:  1 23.789473684210527
strain_various:  1 7.331108535178745
fd_r_thres

glc__D_e_1:  -7.558357688906002
glc__D_e_1:  -5.294343061236217
glc__D_e_1:  -1.2770493600889827
glc__D_e_1:  -1.3326411138581342
glc__D_e_1:  -11.56202693360769
glc__D_e_1:  -10.326471559201337
glc__D_e_1:  -3.2731691668191134
glc__D_e_1:  -6.279516032583157
glc__D_e_1:  -10.701901609835097
glc__D_e_1:  -6.122975993266192
glc__D_e_1:  -7.3825590757866095
glc__D_e_1:  -8.532919374425694
glc__D_e_1:  -10.534923632673621
glc__D_e_1:  -6.624672064537227
glc__D_e_1:  -4.107687347159006
glc__D_e_1:  -7.169625966692201
glc__D_e_1:  -12.927897137664726
glc__D_e_1:  -8.194527099295026
glc__D_e_1:  -5.868176745536023
glc__D_e_1:  -8.438322846403402
glc__D_e_1:  -11.522550480882105
glc__D_e_1:  -8.949647009610732
glc__D_e_1:  -3.2277596092203718
glc__D_e_1:  -4.75872891898697
glc__D_e_1:  -7.463506525544601
glc__D_e_1:  -6.536839994739838
glc__D_e_1:  -3.790674430648442
glc__D_e_1:  -1.4428953448576682
glc__D_e_1:  -7.572085166569498
glc__D_e_1:  -9.988788877927288
glc__D_e_1:  -3.37734652828344

glc__D_e_1:  -1.7859290951944509
glc__D_e_1:  -0.8083662031603522
glc__D_e_1:  -0.04466068004409296
glc__D_e_1:  -0.16173180702419643
glc__D_e_1:  -0.007287385223402332
glc__D_e_1:  -0.2941397221683182
glc__D_e_1:  -1.1387395273361336
glc__D_e_1:  -2.321643202400361
glc__D_e_1:  -0.47301175071539836
glc__D_e_1:  -1.1907493751358968
glc__D_e_1:  -1.1033781931862363
glc__D_e_1:  -0.18431787977931013
glc__D_e_1:  -1.3672215548435376
glc__D_e_1:  -0.688640711107563
glc__D_e_1:  -1.8476525168499385
glc__D_e_1:  -0.024623455783015835
glc__D_e_1:  -0.7819476630490105
glc__D_e_1:  -0.41976405138714346
glc__D_e_1:  -0.20411449655071578
strain_number:  0 3.4210526315789473
strain_various:  0 1.2697197980857704
strain_number:  1 1.5263157894736843
strain_various:  1 0.7517293082676684
fd_r_threshold:  [1.05, 31.887449031454906, 72.77949791254017, 18.440134008017214, 13.268222447278367, 16.709326781408926, 18.551002724380307, 8.114938863791235, 4.418415101534866, 5.111448468986024, 4.4165661671598

glc__D_e_1:  -0.19360562196717107
glc__D_e_1:  -0.07270879919861828
glc__D_e_1:  -0.17573017183637918
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 31.887449031454906, 72.77949791254017, 18.440134008017214, 13.268222447278367, 16.709326781408926, 18.551002724380307, 8.114938863791235, 4.418415101534866, 5.111448468986024, 4.4165661671598055, 5.851394131461686, 4.545409352440136, 3.8842554905622824, 4.7480287926266955, 4.235524376417233, 4.455141723356009, 4.9243083900226745, 6.178888888888888, 4.784722222222222, 6.972222222222222, 7.111111111111111, 2.5, 0.25, 6.25, 4.0]
slip_r:  4.022222222222222
27
glc__D_e_1:  -0.05775299257200833
glc__D_e_1:  -0.051857133142781286
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 31.887449031454906, 72.77949791254017, 18.440134008017214, 13.268222447278367,